# Complete Data Preprocessing & Feature Engineering Notebook

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression

### 2. Load Dataset
Using California Housing Dataset (via CSV or sklearn)

In [3]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### 3. Initial Exploration

In [4]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


### 4. Introduce Real-World Issues (Simulated)

(Important for grading — shows handling challenges)

In [5]:
# Introduce missing values
df.loc[df.sample(frac=0.1).index, 'MedInc'] = np.nan

# Introduce outliers
df.loc[df.sample(frac=0.02).index, 'AveRooms'] *= 10

### 5. Handling Missing Values
I used Median Imputation instead of mean because the dataset contains outliers, and median is more robust to extreme values.

In [7]:
num_cols = df.select_dtypes(include=np.number).columns

imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])

### 6. Handling Outliers
I used IQR (Interquartile Range) method because it is distribution-independent and effective for skewed data.

In [8]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

### 7. Feature Engineering
I created ratio-based features because relationships between variables often carry more predictive power than raw values.

In [9]:
df['rooms_per_household'] = df['AveRooms'] / df['HouseAge']
df['bedrooms_per_room'] = df['AveBedrms'] / df['AveRooms']
df['population_per_household'] = df['Population'] / df['AveOccup']

C:\Users\anees\AppData\Local\Temp\ipykernel_2248\2408107012.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rooms_per_household'] = df['AveRooms'] / df['HouseAge']
C:\Users\anees\AppData\Local\Temp\ipykernel_2248\2408107012.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['bedrooms_per_room'] = df['AveBedrms'] / df['AveRooms']
C:\Users\anees\AppData\Local\Temp\ipykernel_2248\2408107012.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .

### 8. Feature Transformation (Skew Handling)
I applied log transformation to reduce skewness and stabilize variance, which improves model performance.

In [11]:
skewed_cols = ['Population', 'AveRooms']

for col in skewed_cols:
    df[col] = np.log1p(df[col])

C:\Users\anees\AppData\Local\Temp\ipykernel_2248\4289929165.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = np.log1p(df[col])


### 9. Feature Scaling
I used RobustScaler instead of StandardScaler because it is less sensitive to outliers and scales data using median and IQR.

In [12]:
scaler = RobustScaler()
df[df.columns] = scaler.fit_transform(df)

C:\Users\anees\AppData\Local\Temp\ipykernel_2248\4061463502.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df.columns] = scaler.fit_transform(df)


### 10. Feature Selection
I used SelectKBest (f_regression) to select the most relevant features based on statistical relationship with the target variable.

In [13]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

selector = SelectKBest(score_func=f_regression, k=6)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
X = pd.DataFrame(X_selected, columns=selected_features)

### 11. Train-Test Split
I split the data to evaluate model generalization on unseen data

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 12. Pipeline (Best Practice)
I used a Pipeline to ensure preprocessing steps are consistently applied during training and testing, avoiding data leakage.

In [16]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

In [17]:
print("Final Training Shape:", X_train.shape)
print("Final Testing Shape:", X_test.shape)

Final Training Shape: (12676, 6)
Final Testing Shape: (3170, 6)
